In [52]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load dataset.

In [63]:
from contra_seq_dataset import ContraSeqDataset
from torch.utils.data import DataLoader, Subset

import random
import numpy as np
random.seed(666)

anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles.csv'

ds = ContraSeqDataset(anc_path, aug_path)
print(len(ds))

## Dataset splits
# val_size = 5000
# train_size = len(trainset) - val_size
# train_ds, val_ds = random_split(trainset, [train_size, val_size])
test_idc = random.sample(range(len(dset)),1000)
train_idc = [i for i in range(len(dset)) if i not in test_idc]
train_ds = Subset(ds, train_idc)
test_ds = Subset(ds, test_idc)

## Dataloaders
bs = 32
train_loader = DataLoader(train_ds, bs, shuffle=True, num_workers=4, pin_memory=True)
# val_loader = DataLoader(val_ds, bs*2, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, bs*2, num_workers=4, pin_memory=True)

10000


### Model

In [64]:
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import ContraSeqDataset

model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)

samp = ds.__getitem__(0)
print(samp)
# print(samp.shape)
model.forward(**samp)

{'seq': tensor([11, 22, 27,  5, 27, 27, 27,  6, 27, 27, 27, 27, 27,  6, 27,  5, 13, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24]), 'pad_mask': tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  T

(tensor([[-0.2971, -0.3091,  0.3223, -0.4241,  0.1294, -0.2118, -1.3417,  0.6050,
           0.4535,  0.3089,  0.2445, -0.1972, -0.4334,  0.2808, -0.2653,  0.4397,
          -0.0062, -0.1404,  0.6865, -0.4945,  0.3656, -0.5572, -0.0706, -0.1273,
          -0.2416, -0.2220, -0.1871,  0.2165, -0.6635, -0.3437,  0.1361,  0.2581]],
        grad_fn=<AddmmBackward0>),
 tensor([[[-0.1598, -0.1218,  0.3738,  ...,  0.0595, -0.7529, -0.1321],
          [-0.0444,  0.2252,  0.6291,  ..., -0.1786, -0.3229, -0.1864],
          [-0.1885, -0.4663,  0.2429,  ..., -0.1720, -0.6872,  0.1226],
          ...,
          [ 0.1627, -0.3433, -0.1154,  ..., -0.1548,  0.0150,  0.1238],
          [ 0.0011, -0.2630, -0.1050,  ..., -0.2866, -0.3923,  0.1905],
          [ 0.0057, -0.1605, -0.3530,  ..., -0.1934, -0.1918,  0.1600]]],
        grad_fn=<MaskedFillBackward0>))

tensor([[11, 22, 27,  5, 27, 27, 27,  6, 27, 27, 27, 27, 27,  6, 27,  5, 13, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24]])
torch.Size([1, 122])


(tensor([[ 0.6520, -0.0581, -0.0567, -0.2904,  0.2466, -0.0388, -0.0724,  0.1159,
           0.5287,  0.6820,  0.0473, -0.4553, -0.7668, -0.2708,  0.3542, -0.1134,
           0.0210, -0.6296, -0.5421,  0.1035,  0.1307,  0.2567, -0.0173, -0.3370,
           0.0405, -0.3667,  0.2702, -0.0944, -0.8660,  0.0722,  0.3039, -0.5622]],
        grad_fn=<AddmmBackward0>),
 tensor([[[-0.2634, -0.1737, -0.1467,  ...,  0.5163, -0.2057,  0.1557],
          [-0.4892, -0.1173,  0.5145,  ...,  0.5690, -0.2131, -0.0918],
          [-0.2441, -0.0720,  0.0874,  ...,  0.7141,  0.0224,  0.1871],
          ...,
          [-0.2831, -0.3499,  0.0116,  ..., -0.3259, -0.8824, -0.0477],
          [-0.2674, -0.6255, -0.1695,  ..., -0.2209, -0.7549,  0.0368],
          [-0.4125, -0.3722,  0.0227,  ..., -0.2039, -0.5651,  0.0972]]],
        grad_fn=<MaskedFillBackward0>))

In [45]:
print(latent_vec)

tensor([[-0.0192,  0.8394, -0.4852, -0.6510,  0.2627,  0.0186,  0.4255,  0.5447,
         -0.2727,  0.3734, -0.2470,  0.5405, -0.0094, -0.2699, -0.3843,  0.3982,
         -0.1854,  0.0605,  0.3964, -0.6879, -0.5280,  0.5382, -0.3215,  0.1850,
         -0.2392, -0.4115,  0.3611, -0.0533,  0.2659, -0.0712,  0.4078,  0.1780]],
       grad_fn=<AddmmBackward0>)


In [46]:
print(dec_out)

tensor([[[ 0.4088, -0.6478, -0.2075,  ..., -0.3168,  0.6421, -0.3276],
         [ 0.7275, -0.9321, -0.5582,  ...,  0.2210,  0.5681, -0.2803],
         [ 0.2669, -0.8232, -0.7234,  ..., -0.2182,  0.5642, -0.3237],
         ...,
         [-0.1063, -0.4341,  0.0665,  ...,  0.0121,  0.1571, -0.6223],
         [-0.2280, -0.5748,  0.2202,  ..., -0.2608,  0.0111, -0.4930],
         [ 0.0315, -0.3900,  0.0456,  ..., -0.4120,  0.1261, -0.7050]]],
       grad_fn=<MaskedFillBackward0>)


In [38]:
samp = ds.__getitem__(0)
print(samp['anchor'])
print(samp['augs'][:4])

tensor([11, 22, 27,  5, 27, 27, 27,  6, 27, 27, 27, 27, 27,  6, 27,  5, 13, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24])
tensor([[11, 14, 27,  5, 27, 27, 27, 27,  6, 27, 27,  1, 22,  2, 27, 27, 27,  5,
          6, 13, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 2